# Test Results — Part 5: GA4GH VCF → FHIR Genomics Reporting + NW-GMSA R01

This is the fifth notebook in the series. `04-laboratory-report-fhir-from-hl7v2.ipynb`
built a **Laboratory Report** — a narrative PDF wrapped in `DiagnosticReport`/
`DocumentReference`/`Binary`. This notebook builds something different: a **Test
Results** message carrying discrete, structured genomic findings straight out of a
GA4GH VCF file, with no PDF and no `conclusionCode` at all. Section 2 below explains
why these are genuinely different message types, not two ways of building the same
thing.

Two stages:

1. **VCF → FHIR.** Convert `Input/DSS/VCF/igene_example_data.vcf` into `variant`
   `Observation` resources conforming to the [HL7 Genomics Reporting
   IG](https://build.fhir.org/ig/HL7/genomics-reporting/). This reuses the parsing and
   resource-building code `VCFToFHIRVariant.ipynb` already established and validated —
   see that notebook for the field-by-field mapping rationale; it isn't repeated here.
2. **Add order context.** Pull the order metadata (patient, encounter, `ServiceRequest`)
   out of `Input/V2/R01/ctdna9737383222.txt` — the same message
   `04-laboratory-report-fhir-from-hl7v2.ipynb` converted — and use it to assemble a
   `Bundle` (message) whose `DiagnosticReport` and its `variant` `Observation`s
   simultaneously conform to **both** the HL7 Genomics Reporting IG and
   [NW-GMSA](https://nw-gmsa.github.io/en/).

## 1. Where this sits in the lab workflow

[NW-GMSA's own LTW page](https://nw-gmsa.github.io/en/LTW.html) maps its messaging onto
IHE's Laboratory Testing Workflow (LTW) transactions:

```mermaid
sequenceDiagram
    participant EPR as Clinician / EPR<br/>(Order Placer)
    participant LIMS as LIMS / GLH<br/>(Order Filler)
    participant SEQ as Sequencer / bioinformatics pipeline<br/>(Automation Manager)

    EPR->>LIMS: LAB-1 Laboratory Order<br/>(clinical requisition)
    LIMS->>SEQ: LAB-4 Work Order<br/>(specimen ID, requested tests,<br/>analyser/pipeline configuration)
    SEQ->>LIMS: LAB-5 Test Results<br/>(discrete measured/called values)
    LIMS->>EPR: LAB-3 Laboratory Report<br/>(authorised, synthesised report)
```

- **`04-laboratory-report-fhir-from-hl7v2.ipynb` built a `LAB-3`** — the authorised
  report a clinician reads, complete with a narrative PDF and a `conclusionCode` summing
  up the outcome.
- **This notebook builds a `LAB-5`** — the sequencer/bioinformatics pipeline handing
  discrete variant calls back to the LIMS, *before* anyone has synthesised them into a
  report. That's why there's no PDF and no `conclusionCode` here: nobody has written a
  conclusion yet. A `LAB-5` like this one is what a later `LAB-3` would be built *from*.
- **The order that triggered this is more likely a `LAB-4` than a `LAB-1`.** A `LAB-1`
  is the clinical requisition a clinician originally placed; by the time a sequencer is
  producing variant calls, what's actually driving it is the LIMS's own `LAB-4` work
  order to the pipeline (specimen ID, which tests to run, pipeline configuration) — a
  step removed from the original clinical order. We still source our order metadata from
  `ctdna9737383222.txt` (an `ORU^R01`/`LAB-3`-shaped fixture, since that's what this repo
  has) purely for a realistic patient/order identity to attach the results to — not as a
  claim that a `LAB-3` message is what actually triggers a `LAB-5`.

```mermaid
flowchart TB
    subgraph NB04["04: Laboratory Report (LAB-3)"]
        D1["DiagnosticReport<br/>+ conclusionCode<br/>+ presentedForm"]
        O1["Observation<br/>GenomicStudyPanel<br/>(indication + outcome)"]
        B1["Binary (PDF)"]
        DR1["DocumentReference"]
        D1 --> O1
        D1 -->|presentedForm| B1
        DR1 -->|content.attachment| B1
    end
    subgraph NB05["05: Test Results (LAB-5)"]
        D2["DiagnosticReport<br/>NW-GMSA + genomics-reporting<br/>no conclusionCode, no presentedForm"]
        V1["Observation: variant #1"]
        V2["Observation: variant #2"]
        V3["Observation: variant #3"]
        V4["Observation: variant #4"]
        D2 --> V1
        D2 --> V2
        D2 --> V3
        D2 --> V4
    end
```

Both still reuse the same `Patient`/`Encounter`/`ServiceRequest` shape from the order
metadata — only the `DiagnosticReport` and its results differ.

In [1]:
import json
import os
import re
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
IGENE_REPORT_ID_SYSTEM = "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier"
GENOMIC_TEST_DIRECTORY_SYSTEM = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"
IGEAP_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP"
GENOMIC_CLINICAL_INDICATION_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication"
GLH_ODS = "699X0"
GLH_NAME = "NHS North West Genomics"

NWGMSA_PATIENT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Patient"
NWGMSA_SERVICE_REQUEST_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ServiceRequest"
NWGMSA_DIAGNOSTIC_REPORT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/DiagnosticReport"
GENOMICS_VARIANT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"
GENOMICS_REPORT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/genomic-report"
GENOMICS_REPORTING_IG = "hl7.fhir.uv.genomics-reporting#3.0.0"


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    df = df[~df["details"].str.contains("no terminology service")]
    return df


def validate(path, igs, profiles, output):
    args = ["java", "-jar", "validator_cli.jar", str(path), "-version", "4.0.1", "-tx", "n/a"]
    for ig in igs:
        args += ["-ig", ig]
    for profile in profiles:
        args += ["-profile", profile]
    args += ["-output", str(output), "-output-style", "json"]
    subprocess.run(args, capture_output=True)
    with open(output) as f:
        return issues_df(json.load(f))


def validate_resource(resource, igs, profiles):
    # Validate a single (non-bundled) resource - a dev-loop check, as in notebooks 03/04.
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = Path(tmp.name)
    return validate(tmp_path, igs, profiles, Path(str(tmp_path) + "-OperationOutcome.json"))

## Stage 1: VCF → `variant` Observations

The next several cells are `VCFToFHIRVariant.ipynb`'s own header/record parsing, lookup
tables, and `build_observation()` function, unchanged — see that notebook for why each
mapping choice was made. `build_observation()` takes the patient/organisation/date to
attach as parameters rather than hard-coding them, specifically so this notebook can
reuse it against a different (real order) context in Stage 2 below.

### Parse the VCF header → field-to-LOINC mapping

In [2]:
VCF_PATH = Path("Input/DSS/VCF/igene_example_data.vcf")


def parse_vcf_header(path):
    meta_re = re.compile(r'^##(INFO|FORMAT)=<ID=([^,]+),.*?Description="([^"]*)"')
    loinc_re = re.compile(r"(\d{4,5}-\d)")
    fields = {}
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if not line.startswith("##"):
                continue
            m = meta_re.match(line)
            if not m:
                continue
            _, field_id, description = m.groups()
            loinc_codes = loinc_re.findall(description) if "LOINC" in description else []
            fields[field_id] = {"description": description, "loinc": loinc_codes}
    return fields


FIELD_MAP = parse_vcf_header(VCF_PATH)
pd.DataFrame(
    [{"field": k, "loinc": ", ".join(v["loinc"]) or "-", "description": v["description"]} for k, v in FIELD_MAP.items()]
)

,field,loinc,description
0,VARTYPE,-,iGene custom-field variant category (Sequence ...
1,GENE,48018-6,Gene studied [ID] (LOINC 48018-6)
2,CYTOBAND,48001-2,Cytogenetic (chromosome) location (LOINC 48001-2)
3,HGVSC,"51958-7, 48004-6","Transcript reference sequence and DNA change, ..."
4,HGVSP,48005-3,"Amino acid change, p.HGVS (LOINC 48005-3)"
5,HGVSG,81290-9,"Genomic DNA change, g.HGVS, exactly as given i..."
6,CLASS,53037-8,Genetic variation clinical significance [Imp] ...
7,CLASSEVIDENCE,-,Free-text summary of evidence supporting the c...
8,INHERITANCE,-,Inheritance of the variant: Maternal/Paternal/...
9,SVTYPE,-,Type of structural variant


### Parse the VCF data rows

In [3]:
def parse_vcf_records(path):
    records = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            if not line or line.startswith("#"):
                continue
            chrom, pos, vid, ref, alt, qual, filt, info, fmt, sample = line.split("\t")
            info_dict = {}
            for item in info.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    info_dict[k] = v
                else:
                    info_dict[item] = True
            format_dict = dict(zip(fmt.split(":"), sample.split(":")))
            records.append(
                {"chrom": chrom, "pos": int(pos), "id": vid, "ref": ref, "alt": alt, "info": info_dict, "format": format_dict}
            )
    return records


RECORDS = parse_vcf_records(VCF_PATH)
pd.DataFrame(
    [
        {"ID": r["id"], "CHROM": r["chrom"], "POS": r["pos"], "REF": r["ref"], "ALT": r["alt"],
         "VARTYPE": r["info"].get("VARTYPE"), "GENE": r["info"].get("GENE")}
        for r in RECORDS
    ]
)

,ID,CHROM,POS,REF,ALT,VARTYPE,GENE
0,SEQV1,17,41276046,TCT,T,Sequence_Variant,BRCA1
1,ICNV1,15,48797221,C,<DEL>,Intragenic_Copy_Number_Variant,FBN1
2,MCNV1,X,100652796,T,<DEL>,Multigenic_Copy_Number_Variant,None
3,SV1,X,100652796,T,<DEL>,Structural_Variant,None


### Lookup tables

In [4]:
GRCH37_REFSEQ = {"17": "NC_000017.10", "15": "NC_000015.9", "X": "NC_000023.10"}
HGNC_GENE = {"BRCA1": "HGNC:1100", "FBN1": "HGNC:3603"}
SO_TERM = {
    "SNV": ("SO:0001483", "SNV"),
    "deletion": ("SO:0000159", "deletion"),
    "insertion": ("SO:0000667", "insertion"),
    "copy_number_variation": ("SO:0001019", "copy_number_variation"),
}
ALLELIC_STATE = {
    "Heterozygous": ("LA6706-1", "heterozygous"),
    "Homozygous": ("LA6705-3", "homozygous"),
    "Hemizygous": ("LA6707-9", "hemizygous"),
}
INHERITANCE_ORIGIN = {"Maternal": ("LA26320-4", "Maternal")}

### Build `variant` Observation resources

In [5]:
def cc(system=None, code=None, display=None, text=None):
    concept = {}
    if system and code:
        coding = {"system": system, "code": code}
        if display:
            coding["display"] = display
        concept["coding"] = [coding]
    if text:
        concept["text"] = text
    elif display and "coding" not in concept:
        concept["text"] = display
    return concept


def component(loinc_code, loinc_display, **value):
    comp = {"code": {"coding": [{"system": "http://loinc.org", "code": loinc_code, "display": loinc_display}]}}
    comp.update(value)
    return comp


def dna_change_type(record):
    vartype = record["info"].get("VARTYPE", "")
    if "Copy_Number_Variant" in vartype:
        return SO_TERM["copy_number_variation"]
    if vartype == "Structural_Variant":
        return SO_TERM["deletion"] if record["info"].get("SVTYPE") == "DEL" else None
    ref, alt = record["ref"], record["alt"]
    if len(ref) == 1 and len(alt) == 1:
        return SO_TERM["SNV"]
    if len(alt) < len(ref):
        return SO_TERM["deletion"]
    if len(alt) > len(ref):
        return SO_TERM["insertion"]
    return None


def build_observation(record, obs_id, patient_ref, org_ref, effective_date):
    info = record["info"]
    fmt = record["format"]
    vartype = info.get("VARTYPE", "")
    is_structural = vartype in ("Intragenic_Copy_Number_Variant", "Multigenic_Copy_Number_Variant", "Structural_Variant")

    components = []

    gene = info.get("GENE")
    if gene:
        hgnc = HGNC_GENE.get(gene)
        components.append(component("48018-6", "Gene studied [ID]",
                                     valueCodeableConcept=cc("http://www.genenames.org", hgnc, gene) if hgnc else cc(text=gene)))

    if "INHERITANCE" in info:
        components.append(component("48002-0", "Genomic source class [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", "LA6683-2", "Germline")))

    refseq = GRCH37_REFSEQ.get(record["chrom"])
    if refseq:
        components.append(component("48013-7", "Genomic reference sequence [ID]",
                                     valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", refseq)))

    components.append(component("92822-6", "Genomic coordinate system [Type]",
                                 valueCodeableConcept=cc("http://loinc.org", "LA30102-0", "1-based character counting")))
    components.append(component("69547-8", "Genomic ref allele [ID]", valueString=record["ref"]))
    components.append(component("69551-0", "Genomic alt allele [ID]", valueString=record["alt"]))

    so_term = dna_change_type(record)
    if so_term:
        code_val, display = so_term
        components.append(component("48019-4", "DNA change type",
                                     valueCodeableConcept=cc("http://www.sequenceontology.org", code_val, display)))

    hgvsc = info.get("HGVSC")
    if hgvsc:
        transcript_match = re.match(r"(N[MR]_\d+\.\d+)", hgvsc)
        if transcript_match:
            components.append(component("51958-7", "Transcript reference sequence [ID]",
                                         valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", transcript_match.group(1))))
        components.append(component("48004-6", "DNA change (c.HGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsc)))

    hgvsp = info.get("HGVSP")
    if hgvsp:
        components.append(component("48005-3", "Amino acid change (pHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsp)))

    hgvsg = info.get("HGVSG")
    if hgvsg:
        components.append(component("81290-9", "Genomic DNA change (gHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsg)))

    cytoband = info.get("CYTOBAND")
    if cytoband:
        components.append(component("48001-2", "Cytogenetic (chromosome) location", valueCodeableConcept=cc(text=cytoband)))

    classification = info.get("CLASS")
    if classification:
        components.append(component("53037-8", "Genetic variation clinical significance [Imp]",
                                     valueCodeableConcept=cc(text=classification.replace("_", " "))))

    inheritance = info.get("INHERITANCE")
    if inheritance:
        origin = INHERITANCE_ORIGIN.get(inheritance)
        components.append(component("94186-4", "Origin of germline genetic variant [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", *origin) if origin else cc(text=inheritance)))

    end = info.get("END")
    if is_structural and end:
        components.append(component("81302-2", "Structural variant inner start and end",
                                     valueRange={"low": {"value": record["pos"]}, "high": {"value": int(end)}}))
    elif not is_structural:
        components.append(component("81254-5", "Genomic allele start-end", valueRange={"low": {"value": record["pos"]}}))

    vaf = fmt.get("VAF")
    if vaf and vaf != ".":
        components.append(component("81258-6", "Sample variant allelic frequency",
                                     valueQuantity={"value": float(vaf), "unit": "decimal", "system": "http://unitsofmeasure.org"}))

    zyg = fmt.get("ZYG")
    if zyg:
        copy_match = re.search(r"\((\d+)_cop(?:y|ies)\)", zyg)
        if copy_match:
            components.append(component("82155-3", "Genomic structural variant copy number",
                                         valueQuantity={"value": int(copy_match.group(1)), "system": "http://unitsofmeasure.org", "code": "1"}))
        elif zyg in ALLELIC_STATE:
            components.append(component("53034-5", "Allelic state", valueCodeableConcept=cc("http://loinc.org", *ALLELIC_STATE[zyg])))

    return {
        "resourceType": "Observation",
        "id": obs_id,
        "meta": {"profile": [GENOMICS_VARIANT_PROFILE]},
        "status": "final",
        "category": [
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]},
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
        ],
        "code": {"coding": [{"system": "http://loinc.org", "code": "69548-6", "display": "Genetic variant assessment"}]},
        "subject": {"reference": patient_ref},
        "effectiveDateTime": effective_date,
        "performer": [{"reference": org_ref}],
        "valueCodeableConcept": cc("http://loinc.org", "LA9633-4", "Present"),
        "method": cc("http://loinc.org", "LA26398-0", "Sequencing"),
        "component": components,
    }

Prove the conversion in isolation first, exactly as `VCFToFHIRVariant.ipynb` does
— a placeholder subject/performer, validated only against the Genomics Reporting IG.

In [6]:
placeholder_patient_ref = "urn:uuid:placeholder-patient"
placeholder_org_ref = "urn:uuid:placeholder-org"

placeholder_observations = [
    build_observation(record, f"placeholder-{record['id'].lower()}", placeholder_patient_ref, placeholder_org_ref, "2026-08-15")
    for record in RECORDS
]

print(json.dumps(placeholder_observations[0], indent=2))

{
  "resourceType": "Observation",
  "id": "placeholder-seqv1",
  "meta": {
    "profile": [
      "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "laboratory"
        }
      ]
    },
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0074",
          "code": "GE"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "69548-6",
        "display": "Genetic variant assessment"
      }
    ]
  },
  "subject": {
    "reference": "urn:uuid:placeholder-patient"
  },
  "effectiveDateTime": "2026-08-15",
  "performer": [
    {
      "reference": "urn:uuid:placeholder-org"
    }
  ],
  "valueCodeableConcept": {
    "coding": [
      {
        "system": "http://loinc.org",
        "co

In [7]:
rows = []
for obs in placeholder_observations:
    df = validate_resource(obs, [GENOMICS_REPORTING_IG], [GENOMICS_VARIANT_PROFILE])
    df.insert(0, "observation", obs["id"])
    rows.append(df)

pd.concat(rows, ignore_index=True)

,observation,severity,code,details,expression
0,placeholder-seqv1,information,informational,This element does not match any known slice de...,[Observation.component[11]]
1,placeholder-seqv1,information,code-invalid,Binding for path Observation.component[2].valu...,[Observation.component[2].value.ofType(Codeabl...
2,placeholder-seqv1,information,code-invalid,Binding for path Observation.component[7].valu...,[Observation.component[7].value.ofType(Codeabl...
3,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[10].value.ofType(Codeab...
4,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[10].value.ofType(Codeab...
5,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[9].value.ofType(Codeabl...
6,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[9].value.ofType(Codeabl...
7,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[8].value.ofType(Codeabl...
8,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Observation.component[6].value.ofType(Codeabl...
9,placeholder-seqv1,warning,not-found,A definition for CodeSystem 'http://www.ncbi.n...,[Observation.component[7].value.ofType(Codeabl...


## Stage 2: order metadata → a combined R01 message

`ctdna9737383222.txt` is the same message `04-laboratory-report-fhir-from-hl7v2.ipynb`
parsed in full — see that notebook for the segment-by-segment mapping rationale. Here we
only pull out what an order/results message needs: nothing about the PDF/outcome `OBX`
segments that notebook also used, since this message doesn't carry either.

In [8]:
def component_field(field, index, default=""):
    parts = field.split("^")
    return parts[index].strip() if index < len(parts) else default


with open("Input/V2/R01/ctdna9737383222.txt", newline="") as f:
    raw_v2 = f.read()

segments = [s for s in raw_v2.replace("\r\n", "\r").split("\r") if s]
by_segment = {}
for segment in segments:
    fields = segment.split("|")
    by_segment.setdefault(fields[0], []).append(fields)

pid = by_segment["PID"][0]
pv1 = by_segment["PV1"][0]
orc = by_segment["ORC"][0]
obr = by_segment["OBR"][0]
nte = by_segment["NTE"][0]

report_row = {
    "nhs_number": pid[2],
    "mrn": component_field(pid[3], 0),
    "mrn_assigner_ods": component_field(pid[3], 3),
    "family_name": component_field(pid[5], 0),
    "given_name": component_field(pid[5], 1),
    "birth_date": datetime.strptime(pid[7], "%Y%m%d").strftime("%Y-%m-%d"),
    "sex": pid[8],
    "postcode": component_field(pid[11], 4),
    "account_number": pv1[19],
    "placer_order_number": orc[2],
    "ordering_org_name": component_field(orc[21], 0),
    "ordering_org_ods": component_field(orc[21], 2),
    "filler_report_number": obr[3],
    "test_code_local": component_field(obr[4], 0),
    "test_description": component_field(obr[4], 1),
    "report_datetime": datetime.strptime(obr[22], "%Y%m%d%H%M%S").strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "test_directory_code": nte[3].split("=")[0],
    "clinical_indication_code": nte[3].split("=")[0].split(".")[0],
}
report_row

{'nhs_number': '9737383222',
 'mrn': 'RXR0817610',
 'mrn_assigner_ods': 'RR8',
 'family_name': 'LEEDS',
 'given_name': 'Rob',
 'birth_date': '1978-01-17',
 'sex': 'M',
 'postcode': 'LS1 3EX',
 'account_number': 'SP26-01847',
 'placer_order_number': '1234-RR8',
 'ordering_org_name': 'Leeds Teaching Hospitals NHS Trust',
 'ordering_org_ods': 'RR8',
 'filler_report_number': 'T26-59X2',
 'test_code_local': 'ctDNA_M4',
 'test_description': 'PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, RET, ALK, NTRK1, NTRK2, NTRK3, MET exon',
 'report_datetime': '2026-07-14T15:59:16+00:00',
 'test_directory_code': 'M4.14',
 'clinical_indication_code': 'M4'}

### Patient, Encounter, ServiceRequest, and an Organization for the GLH

Same shapes as `04-laboratory-report-fhir-from-hl7v2.ipynb` (see that notebook for the
per-field rationale), with one addition: an actual `Organization` resource for the GLH.
NW-GMSA resources reference organisations by logical identifier alone; the Genomics
Reporting IG's `variant.performer` just wants a plain `Reference`, so here the GLH gets
both — a real bundle entry to satisfy the latter, referenced by identifier *and*
`urn:uuid` everywhere, satisfying the former too.

In [9]:
patient_fullurl = f"urn:uuid:{uuid4()}"
patient = {
    "resourceType": "Patient",
    "identifier": [
        {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["mrn_assigner_ods"]}},
         "type": {"coding": [{"system": V2_0203, "code": "MR"}]}, "value": report_row["mrn"]},
    ],
    "name": [{"family": report_row["family_name"], "given": [report_row["given_name"]]}],
    "gender": {"M": "male", "F": "female"}.get(report_row["sex"], "unknown"),
    "birthDate": report_row["birth_date"],
    "address": [{"postalCode": report_row["postcode"]}],
}

encounter_fullurl = f"urn:uuid:{uuid4()}"
encounter = {
    "resourceType": "Encounter",
    "status": "finished",
    "class": {"system": "http://terminology.hl7.org/CodeSystem/v3-ActCode", "code": "OBSENC"},
    "identifier": [{"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
}

organization_fullurl = f"urn:uuid:{uuid4()}"
organization = {
    "resourceType": "Organization",
    "identifier": [{"system": ODS_SYSTEM, "value": GLH_ODS}],
    "name": GLH_NAME,
}

service_request_fullurl = f"urn:uuid:{uuid4()}"
service_request = {
    "resourceType": "ServiceRequest",
    "status": "completed",
    "intent": "order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"]}]},
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": report_row["placer_order_number"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]},
    ],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "requester": {"display": report_row["ordering_org_name"],
                  "identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}, "type": "Organization"},
    "reasonCode": [{"coding": [{"system": GENOMIC_CLINICAL_INDICATION_SYSTEM, "code": report_row["clinical_indication_code"]}]}],
    "encounter": {"reference": encounter_fullurl, "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}},
}

print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "completed",
  "intent": "order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "M4.14"
      }
    ]
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RR8"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "1234-RR8"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "system": "https://fhir.nwgenomics.nhs.uk

### Re-run `build_observation()` against the real patient

Same VCF, same function — just a real `patient_ref`/`org_ref`/`effective_date` this
time. Profile conformance doesn't depend on *whose* data it is, so there's no need to
re-validate these individually; Stage 1 already proved the shape.

In [10]:
observations = [
    build_observation(record, f"ctdna9737383222-{record['id'].lower()}", patient_fullurl, organization_fullurl, report_row["report_datetime"])
    for record in RECORDS
]

len(observations)

4

### DiagnosticReport — conforming to both IGs at once

`meta.profile` lists both profiles. `category` (`GE`) and one of the three `code`
codings (LOINC `51969-4`, "Genetic analysis report") are fixed requirements of the
Genomics Reporting IG's own
[`genomic-report`](https://build.fhir.org/ig/HL7/genomics-reporting/StructureDefinition-genomic-report.html)
profile; the other two `code` codings (`IGEAP`, Genomic Test Directory) are what
NW-GMSA's own `DiagnosticReport` profile expects — a profile stacks requirements rather
than replacing them, so both fit in the same `CodeableConcept.coding[]`. `result`
references all four `variant` Observations. There's deliberately no `presentedForm` and
no `conclusionCode` — see section 1.

In [11]:
diagnostic_report_fullurl = f"urn:uuid:{uuid4()}"
diagnostic_report = {
    "resourceType": "DiagnosticReport",
    "meta": {"profile": [NWGMSA_DIAGNOSTIC_REPORT_PROFILE, GENOMICS_REPORT_PROFILE]},
    "status": "final",
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]}],
    "code": {
        "coding": [
            {"system": IGEAP_SYSTEM, "code": report_row["test_code_local"], "display": report_row["test_description"]},
            {"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"], "display": report_row["test_description"]},
            {"system": "http://loinc.org", "code": "51969-4", "display": "Genetic analysis report"},
        ]
    },
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "encounter": {"reference": encounter_fullurl, "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}},
    "effectiveDateTime": report_row["report_datetime"],
    "issued": report_row["report_datetime"],
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]}
    ],
    "basedOn": [{"reference": service_request_fullurl, "type": "ServiceRequest",
                 "identifier": {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
                                "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": report_row["placer_order_number"]}}],
    "result": [{"reference": f"urn:uuid:{o['id']}", "type": "Observation"} for o in observations],
    "performer": [{"display": GLH_NAME, "reference": organization_fullurl,
                   "identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"}],
}

print(json.dumps(diagnostic_report, indent=2)[:2000])

{
  "resourceType": "DiagnosticReport",
  "meta": {
    "profile": [
      "https://fhir.nwgenomics.nhs.uk/StructureDefinition/DiagnosticReport",
      "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/genomic-report"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0074",
          "code": "GE"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP",
        "code": "ctDNA_M4",
        "display": "PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, RET, ALK, NTRK1, NTRK2, NTRK3, MET exon"
      },
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "M4.14",
        "display": "PACKAGE: M4.14

In [12]:
validate_resource(diagnostic_report, ["package.tgz", GENOMICS_REPORTING_IG], [NWGMSA_DIAGNOSTIC_REPORT_PROFILE, GENOMICS_REPORT_PROFILE])

,severity,code,details,expression
0,error,structure,DiagnosticReport.presentedForm: minimum requir...,[DiagnosticReport]
1,information,informational,This element does not match any known slice de...,[DiagnosticReport.code.coding[0]]
2,information,informational,This element does not match any known slice de...,[DiagnosticReport.code.coding[2]]
4,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[0]]
5,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[1]]
6,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[2]]
7,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[3]]


## Assemble the Bundle

`MessageHeader.focus` points at the `DiagnosticReport`, same as
`04-laboratory-report-fhir-from-hl7v2.ipynb`. `Bundle.identifier`/`.timestamp` are
mandatory on the
[`Bundle` (message) profile](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage.html),
as in every message `Bundle` built across this series.

In [13]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "R01"},
    "sender": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
    "destination": [{"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/EPR",
                      "receiver": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}}}],
    "source": {"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/HIVE", "software": "NW GLH"},
    "focus": [{"reference": diagnostic_report_fullurl}],
}

results_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": (
        [{"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
         {"fullUrl": patient_fullurl, "resource": patient},
         {"fullUrl": encounter_fullurl, "resource": encounter},
         {"fullUrl": organization_fullurl, "resource": organization},
         {"fullUrl": service_request_fullurl, "resource": service_request}]
        + [{"fullUrl": f"urn:uuid:{o['id']}", "resource": o} for o in observations]
        + [{"fullUrl": diagnostic_report_fullurl, "resource": diagnostic_report}]
    ),
}

results_filename = "ctdna9737383222-testresults.json"
with open("Input/FHIR/R01/" + results_filename, "w") as f:
    json.dump(results_bundle, f, indent=2)

print("Saved Input/FHIR/R01/" + results_filename)

Saved Input/FHIR/R01/ctdna9737383222-testresults.json


Validate the whole `Bundle` against both `DiagnosticReport` profiles at once —
`-bundle` rules are repeatable in a single validator invocation.

In [14]:
outcome_path = Path("Results/FHIR/R01/" + results_filename + "-OperationOutcome.json")
subprocess.run(
    [
        "java", "-jar", "validator_cli.jar", "Input/FHIR/R01/" + results_filename,
        "-version", "4.0.1", "-ig", "package.tgz", "-ig", GENOMICS_REPORTING_IG,
        "-bundle", "DiagnosticReport:0", NWGMSA_DIAGNOSTIC_REPORT_PROFILE,
        "-bundle", "DiagnosticReport:0", GENOMICS_REPORT_PROFILE,
        "-tx", "n/a",
        "-output", str(outcome_path), "-output-style", "json",
    ],
    capture_output=True,
)

with open(outcome_path) as f:
    issues_df(json.load(f))

## Summary and what's next

Starting from a GA4GH VCF and one line of order metadata already used elsewhere in this
series, we built `variant` `Observation`s conforming to the HL7 Genomics Reporting IG,
then combined them with NW-GMSA order context into a single `DiagnosticReport`/`Bundle`
that satisfies both IGs at once — a `LAB-5` Test Results message, not the `LAB-3`
Laboratory Report `04-laboratory-report-fhir-from-hl7v2.ipynb` built.

**Next in this series:** adding these Test Results to an R01 Laboratory Report to
produce a **EU Laboratory Report** FHIR *Document* — combining the discrete `variant`
data this notebook produced with the narrative report `04` produced, into the single
document `Bundle` shape the [EU Laboratory
IG](http://hl7.eu/fhir/laboratory/) defines. This matters beyond being the next logical
step in this series: the **EU Laboratory Report is the format NHS England's Unified
Genomic Record (UGR) Phase II expects.**